# Sesión 3: Consultas de Selección con Tablas Relacionadas (Parte II)
## RIGHT JOIN y FULL OUTER JOIN en SQL

**Módulo:** Fundamentos de Programación Python para el Análisis de Datos

**Contenido:**
- RIGHT JOIN: Conservación de todos los registros de la tabla derecha
- FULL OUTER JOIN: Integración total sin pérdidas
- Limitaciones técnicas según BD
- Emulación en motores sin soporte
- Actividades prácticas avanzadas

## Configuración Inicial

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Librerías importadas correctamente")
print(f"sqlite3 versión: {sqlite3.version}")
print(f"pandas versión: {pd.__version__}")

## SLIDE 3: Desafío Inicial

**Contexto:** Eres parte del equipo de análisis en una consultora que trabaja con datos de estudiantes y sus rendimientos académicos. Los datos de los estudiantes se encuentran en una tabla y las evaluaciones en otra.

**Problema detectado:** Algunas evaluaciones fueron cargadas sin estar vinculadas a ningún estudiante, y algunos estudiantes aún no tienen evaluaciones registradas.

**Pregunta:** ¿Cómo obtener información completa incluyendo estudiantes sin evaluaciones Y evaluaciones sin estudiante?

In [ ]:
# Crear conexión a BD
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("Conexión a base de datos SQLite establecida")
print("\nNota: SQLite no soporta FULL OUTER JOIN nativamente")
print("Lo emularemos con UNION de LEFT JOIN + RIGHT JOIN")

## SLIDE 4-5: RIGHT JOIN - Concepto

**¿Qué es RIGHT JOIN?**
Un RIGHT JOIN devuelve todas las filas de la tabla derecha (tabla2), combinándolas con las filas correspondientes de la izquierda. Si no hay coincidencia, aparecen valores NULL en las columnas de la tabla izquierda.

In [ ]:
# Crear tabla ESTUDIANTES
cursor.execute('''
CREATE TABLE estudiantes (
    id_estudiante INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    email TEXT,
    carrera TEXT,
    semestre INTEGER
)
''')

# Insertar datos de estudiantes
estudiantes_data = [
    (1, 'Ana García', 'ana.garcia@uni.edu', 'Ingeniería', 3),
    (2, 'Bruno López', 'bruno.lopez@uni.edu', 'Administración', 2),
    (3, 'Carlos Martín', 'carlos.martin@uni.edu', 'Ingeniería', 3),
    (4, 'Diana Ruiz', 'diana.ruiz@uni.edu', 'Derecho', 4),
    (5, 'Elena Sánchez', 'elena.sanchez@uni.edu', 'Ingeniería', 2)
]

cursor.executemany(
    'INSERT INTO estudiantes VALUES (?, ?, ?, ?, ?)',
    estudiantes_data
)

conn.commit()
print("Tabla 'estudiantes' creada con 5 registros")

df_est = pd.read_sql('SELECT * FROM estudiantes', conn)
print("\nContenido:")
print(df_est)

In [ ]:
# Crear tabla EVALUACIONES
cursor.execute('''
CREATE TABLE evaluaciones (
    id_evaluacion INTEGER PRIMARY KEY,
    id_estudiante INTEGER,
    fecha_evaluacion DATE NOT NULL,
    materia TEXT,
    calificacion DECIMAL(3, 1),
    FOREIGN KEY(id_estudiante) REFERENCES estudiantes(id_estudiante)
)
''')

# Insertar datos de evaluaciones
# Nota: id_evaluacion 5 tiene id_estudiante NULL (evaluación huérfana)
evaluaciones_data = [
    (1, 1, '2024-02-15', 'Cálculo I', 8.5),
    (2, 1, '2024-03-10', 'Álgebra', 9.0),
    (3, 2, '2024-02-20', 'Contabilidad', 7.5),
    (4, 3, '2024-03-05', 'Cálculo I', 8.0),
    (5, None, '2024-03-12', 'Física', 7.0),  # Evaluación sin estudiante
    (6, 4, '2024-02-28', 'Derecho Penal', 8.5),
    (7, None, '2024-03-15', 'Química', 6.5)   # Otra evaluación huérfana
]

cursor.executemany(
    'INSERT INTO evaluaciones VALUES (?, ?, ?, ?, ?)',
    evaluaciones_data
)

conn.commit()
print("Tabla 'evaluaciones' creada con 7 registros")
print("\n⚠ Nota: Dos evaluaciones (id 5 y 7) no tienen estudiante asociado")

df_eval = pd.read_sql('SELECT * FROM evaluaciones', conn)
print("\nContenido:")
print(df_eval)

## SLIDE 5-6: RIGHT JOIN - Sintaxis y Aplicación Práctica

**Sintaxis:**
```sql
SELECT *
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
```

**Uso:** Útil cuando se debe auditar la carga de evaluaciones y asegurar que ninguna quede huérfana.

In [ ]:
# RIGHT JOIN: Todas las evaluaciones, con info del estudiante si existe
query_right_join = '''
SELECT 
    e.id_estudiante,
    e.nombre,
    ev.id_evaluacion,
    ev.fecha_evaluacion,
    ev.materia,
    ev.calificacion
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
ORDER BY ev.id_evaluacion
'''

df_right = pd.read_sql(query_right_join, conn)
print("RESULTADO RIGHT JOIN:")
print(f"Total de registros: {len(df_right)}")
print("\nDatos:")
print(df_right.to_string())
print("\n** Nota: Las evaluaciones 5 y 7 tienen NULL en columnas de estudiantes **")
print("** Esto permite detectar evaluaciones sin estudiante asignado **")

In [ ]:
# Identificar evaluaciones huérfanas
print("\n" + "="*70)
print("CASO DE USO: Encontrar evaluaciones sin estudiante asignado")
print("="*70 + "\n")

query_huerfanas = '''
SELECT 
    ev.id_evaluacion,
    ev.fecha_evaluacion,
    ev.materia,
    ev.calificacion
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
WHERE e.id_estudiante IS NULL
'''

df_huerfanas = pd.read_sql(query_huerfanas, conn)
print(f"Evaluaciones huérfanas encontradas: {len(df_huerfanas)}\n")
print(df_huerfanas.to_string())
print("\n💡 Estas evaluaciones necesitan ser asignadas a un estudiante en el sistema")

## SLIDE 7-8: FULL OUTER JOIN - Concepto y Aplicación

**¿Qué es FULL OUTER JOIN?**
Un FULL OUTER JOIN conserva todos los registros de ambas tablas. Cuando hay coincidencia entre claves, las filas se combinan. Cuando no, los valores de una tabla aparecen como NULL.

**Aplicación práctica:**
Permite construir un reporte completo, detectando tanto estudiantes sin evaluación como evaluaciones sin asignación. Ideal para diagnósticos y conciliaciones de datos.

In [ ]:
# En SQLite, emular FULL OUTER JOIN con UNION
print("\n" + "="*70)
print("EMULACIÓN DE FULL OUTER JOIN EN SQLITE")
print("="*70)
print("\nNota: SQLite no soporta FULL OUTER JOIN nativamente")
print("Usamos UNION de LEFT JOIN + RIGHT JOIN excluyendo coincidencias\n")

# FULL OUTER JOIN emulado
query_full_outer = '''
-- Parte 1: LEFT JOIN (estudiantes sin evaluación)
SELECT 
    e.id_estudiante,
    e.nombre,
    ev.id_evaluacion,
    ev.fecha_evaluacion,
    ev.materia,
    ev.calificacion
FROM estudiantes e
LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante

UNION

-- Parte 2: Evaluaciones sin estudiante (RIGHT EXCLUDE)
SELECT 
    e.id_estudiante,
    e.nombre,
    ev.id_evaluacion,
    ev.fecha_evaluacion,
    ev.materia,
    ev.calificacion
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
WHERE e.id_estudiante IS NULL
'''

df_full = pd.read_sql(query_full_outer, conn)
print("RESULTADO FULL OUTER JOIN (emulado):")
print(f"Total de registros: {len(df_full)}")
print("\nDatos:")
print(df_full.sort_values(by=['id_estudiante', 'id_evaluacion'], 
                          na_position='last').to_string())
print("\n** Combina TODO: estudiantes con evaluaciones + sin evaluaciones + evaluaciones huérfanas **")

## SLIDE 9: Limitación Técnica - Soporte FULL OUTER JOIN

**Motores que soportan FULL OUTER JOIN:**
- PostgreSQL ✓
- SQL Server ✓
- Oracle ✓

**Motores que NO lo soportan:**
- MySQL ✗
- SQLite ✗

**Solución:** Emular con UNION de LEFT JOIN + RIGHT JOIN

In [ ]:
# Demostración: Comparación de emulaciones
print("="*70)
print("COMPARACIÓN: LEFT JOIN vs RIGHT JOIN vs FULL OUTER (emulado)")
print("="*70 + "\n")

# LEFT JOIN
query_left = '''
SELECT e.nombre, ev.materia
FROM estudiantes e
LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
'''
df_left_comp = pd.read_sql(query_left, conn)
print(f"LEFT JOIN: {len(df_left_comp)} registros (todos estudiantes)")
print(df_left_comp.to_string())

# RIGHT JOIN
query_right = '''
SELECT e.nombre, ev.materia
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
'''
df_right_comp = pd.read_sql(query_right, conn)
print(f"\nRIGHT JOIN: {len(df_right_comp)} registros (todas evaluaciones)")
print(df_right_comp.to_string())

## SLIDE 10: Comparación Visual de Todas las Combinaciones JOIN

In [ ]:
# Tabla comparativa
print("\n" + "="*100)
print("COMPARACIÓN COMPLETA: INNER vs LEFT vs RIGHT vs FULL OUTER JOIN")
print("="*100 + "\n")

comparison = {
    'Tipo JOIN': ['INNER JOIN', 'LEFT JOIN', 'RIGHT JOIN', 'FULL OUTER JOIN'],
    'Registros Mantenidos': [
        'Solo coincidencias',
        'Todos de tabla izquierda',
        'Todos de tabla derecha',
        'Todos de ambas tablas'
    ],
    'Coincidencias Requeridas': [
        'Sí',
        'Opcional',
        'Opcional',
        'Opcional'
    ],
    'Genera NULLs': [
        'No',
        'Sí (tabla derecha)',
        'Sí (tabla izquierda)',
        'Sí (ambas)'
    ],
    'En Nuestro Ejemplo': [
        f'{len(pd.read_sql("SELECT * FROM estudiantes e INNER JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante", conn))} registros',
        f'{len(pd.read_sql("SELECT * FROM estudiantes e LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante", conn))} registros',
        f'{len(df_right)} registros',
        f'{len(df_full)} registros'
    ]
}

df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_string(index=False))
print("\n" + "="*100)

## SLIDE 11: Buenas Prácticas

**Recomendaciones:**
1. Clarificar el objetivo del análisis antes de elegir el tipo de combinación
2. Documentar el comportamiento esperado respecto a los NULL
3. Validar si el sistema de base de datos soporta el tipo de JOIN requerido

In [ ]:
# Buena práctica 1: Usar alias y comentarios claros
print("✓ BUENA PRÁCTICA 1: Código documentado y claro\n")

query_buena = '''
-- Auditoría de datos: Detectar inconsistencias en carga de evaluaciones
SELECT 
    COALESCE(e.id_estudiante, -1) as id_estudiante,
    e.nombre,
    ev.id_evaluacion,
    ev.materia,
    CASE 
        WHEN e.id_estudiante IS NULL THEN 'EVALUACIÓN HUÉRFANA'
        WHEN ev.id_evaluacion IS NULL THEN 'ESTUDIANTE SIN EVALUACIÓN'
        ELSE 'OK'
    END as estado
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
ORDER BY estado
'''

df_audita = pd.read_sql(query_buena, conn)
print(df_audita.to_string())
print("\n💡 Usar CASE WHEN para clasificar el estado de los datos")

In [ ]:
# Buena práctica 2: Validar datos después del JOIN
print("\n✓ BUENA PRÁCTICA 2: Validación de datos\n")

print("Conteos de validación:")
print("-" * 50)

# Contar cada tipo de registro
query_valida = '''
SELECT 
    CASE 
        WHEN e.id_estudiante IS NULL THEN 'HUÉRFANAS'
        WHEN ev.id_evaluacion IS NULL THEN 'SIN EVALUACIÓN'
        ELSE 'VINCULADAS'
    END as tipo,
    COUNT(*) as cantidad
FROM estudiantes e
FULL OUTER JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
GROUP BY tipo
'''

# Para SQLite, usar la emulación
query_valida_sqlite = '''
SELECT 
    CASE 
        WHEN e.id_estudiante IS NULL THEN 'HUÉRFANAS'
        WHEN ev.id_evaluacion IS NULL THEN 'SIN EVALUACIÓN'
        ELSE 'VINCULADAS'
    END as tipo,
    COUNT(*) as cantidad
FROM (SELECT e.id_estudiante, e.nombre, ev.id_evaluacion
      FROM estudiantes e
      LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
      UNION
      SELECT e.id_estudiante, e.nombre, ev.id_evaluacion
      FROM estudiantes e
      RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
      WHERE e.id_estudiante IS NULL) AS datos
GROUP BY tipo
'''

df_valida = pd.read_sql(query_valida_sqlite, conn)
print(df_valida.to_string())
print("\n💡 Verificar la distribución para detectar anomalías")

## SLIDE 12-16: Actividad Guiada 1 - Combinando información de estudiantes y evaluaciones

**Objetivo:** Aplicar RIGHT JOIN y FULL OUTER JOIN para integrar registros de dos tablas relacionadas, identificando coincidencias y ausencias de datos.

In [ ]:
print("\n" + "="*70)
print("ACTIVIDAD GUIADA 1: Auditoría de Datos Educativos")
print("="*70 + "\n")

print("📋 CONTEXTO: Instituto educativo que detectó evaluaciones cargadas sin")
print("            estudiante asignado. Necesitamos hacer auditoría completa.\n")

print("✓ CONSULTA 1: Usar RIGHT JOIN para encontrar todas las evaluaciones")
print("-" * 70)

query_act1_1 = '''
SELECT 
    ev.id_evaluacion,
    e.nombre as estudiante,
    ev.materia,
    ev.fecha_evaluacion,
    ev.calificacion
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
ORDER BY ev.id_evaluacion
'''

print("\nSQL:")
print(query_act1_1)
print("\nResultado:")
df_act1_1 = pd.read_sql(query_act1_1, conn)
print(df_act1_1.to_string())
print(f"\n📊 Total: {len(df_act1_1)} evaluaciones (todas mostradas)")

In [ ]:
print("\n✓ CONSULTA 2: Usar FULL OUTER JOIN para reporte completo")
print("-" * 70)

query_act1_2 = '''
-- Emulación FULL OUTER JOIN en SQLite
SELECT 
    COALESCE(e.id_estudiante, -1) as id_estudiante,
    e.nombre,
    ev.id_evaluacion,
    ev.materia,
    ev.calificacion
FROM estudiantes e
LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante

UNION

SELECT 
    COALESCE(e.id_estudiante, -1) as id_estudiante,
    e.nombre,
    ev.id_evaluacion,
    ev.materia,
    ev.calificacion
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
WHERE e.id_estudiante IS NULL
'''

print("\nSQL (con UNION de LEFT + RIGHT):")
print(query_act1_2[:150] + "...\n")

print("Resultado:")
df_act1_2 = pd.read_sql(query_act1_2, conn)
print(df_act1_2.to_string())
print(f"\n📊 Total: {len(df_act1_2)} registros (estudiantes + evaluaciones + huérfanas)")

In [ ]:
print("\n✓ ANÁLISIS: Diferencias observadas")
print("-" * 70 + "\n")

print("¿Qué diferencias observas entre ambas combinaciones?\n")

print("RIGHT JOIN:")
print(f"  - Total: {len(df_act1_1)} registros")
print("  - Incluye: Todas las evaluaciones")
print(f"  - NULL en nombre: {df_act1_1['estudiante'].isna().sum()} evaluaciones huérfanas")
print("  - No incluye: Estudiantes sin evaluaciones")

print("\nFULL OUTER (emulado):")
print(f"  - Total: {len(df_act1_2)} registros")
print("  - Incluye: Todas las evaluaciones + estudiantes sin evaluaciones")
print("  - NULL en evaluaciones: Estudiantes sin evaluación")
print("  - NULL en estudiante: Evaluaciones huérfanas")

print("\n¿Qué implican los valores NULL en cada caso?")
print("  RIGHT: NULL significa evaluación sin estudiante (dato huérfano)")
print("  FULL: NULL puede significar estudante sin evaluación O evaluación sin estudiante")

## SLIDE 18-22: Actividad Práctica Autónoma - Integración de datos educativos

**Objetivo:** Aplicar RIGHT JOIN y FULL OUTER JOIN de manera autónoma, analizando cómo se comportan frente a claves faltantes.

In [ ]:
print("\n" + "="*70)
print("ACTIVIDAD PRÁCTICA AUTÓNOMA: Análisis de Integridad de Datos")
print("="*70 + "\n")

print("📋 CONTEXTO: Base de datos institucional con estudiantes y evaluaciones.")
print("            Algunos estudiantes aún no rinden evaluaciones.")
print("            Algunas evaluaciones fueron cargadas sin estudiante.\n")

print("Ejercicio 1: RIGHT JOIN - Todas las evaluaciones")
print("-" * 70)

query_ej1 = '''
SELECT 
    ev.id_evaluacion,
    COALESCE(e.nombre, 'SIN ESTUDIANTE') as estudiante,
    COALESCE(e.carrera, '-') as carrera,
    ev.materia,
    ev.calificacion
FROM estudiantes e
RIGHT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
ORDER BY ev.id_evaluacion
'''

df_ej1 = pd.read_sql(query_ej1, conn)
print("\nResultado (con COALESCE para mejorar legibilidad):")
print(df_ej1.to_string())
print(f"\nInterpretación: {len(df_ej1)} evaluaciones registradas")
print(f"  - Asignadas: {(df_ej1['estudiante'] != 'SIN ESTUDIANTE').sum()}")
print(f"  - Huérfanas: {(df_ej1['estudiante'] == 'SIN ESTUDIANTE').sum()}")

In [ ]:
print("\nEjercicio 2: FULL OUTER JOIN - Reporte completo de integridad")
print("-" * 70)

query_ej2 = '''
-- FULL OUTER emulado
SELECT 
    COALESCE(e.id_estudiante, -999) as id_est,
    e.nombre,
    e.carrera,
    COUNT(ev.id_evaluacion) as total_evaluaciones,
    ROUND(AVG(ev.calificacion), 2) as promedio_calificacion
FROM estudiantes e
LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
GROUP BY e.id_estudiante, e.nombre, e.carrera
'''

df_ej2 = pd.read_sql(query_ej2, conn)
print("\nResultado (Resumen por estudiante):")
print(df_ej2.to_string())
print(f"\nInterpretación:")
print(f"  - Total estudiantes: {len(df_ej2)}")
print(f"  - Sin evaluaciones: {(df_ej2['total_evaluaciones'] == 0).sum()}")
print(f"  - Con evaluaciones: {(df_ej2['total_evaluaciones'] > 0).sum()}")

In [ ]:
print("\nEjercicio 3: Análisis de casos problemáticos")
print("-" * 70)

# Identificar anomalías
print("\na) Estudiantes sin evaluaciones:")
query_sin_eval = '''
SELECT e.nombre, e.carrera
FROM estudiantes e
LEFT JOIN evaluaciones ev ON e.id_estudiante = ev.id_estudiante
WHERE ev.id_evaluacion IS NULL
'''
df_sin = pd.read_sql(query_sin_eval, conn)
print(df_sin.to_string())

print("\nb) Evaluaciones sin estudiante:")
query_eval_sin_est = '''
SELECT ev.materia, ev.calificacion, ev.fecha_evaluacion
FROM evaluaciones ev
LEFT JOIN estudiantes e ON ev.id_estudiante = e.id_estudiante
WHERE e.id_estudiante IS NULL
'''
df_huerfanas = pd.read_sql(query_eval_sin_est, conn)
print(df_huerfanas.to_string())

print("\n📊 REPORTE DE INTEGRIDAD:")
print("-" * 70)
print(f"Estudiantes registrados: {len(pd.read_sql('SELECT COUNT(*) FROM estudiantes', conn))}")
print(f"Evaluaciones registradas: {len(pd.read_sql('SELECT COUNT(*) FROM evaluaciones', conn))}")
print(f"Estudiantes sin evaluación: {len(df_sin)}")
print(f"Evaluaciones huérfanas: {len(df_huerfanas)}")
print(f"Vinculi correctos: {len(pd.read_sql('SELECT COUNT(*) FROM evaluaciones WHERE id_estudiante IS NOT NULL', conn))}")

## SLIDE 23: Resumen de la Sesión

In [ ]:
print("\n" + "="*70)
print("RESUMEN - SESIÓN 3")
print("="*70 + "\n")

summary = """
✓ PROPÓSITO Y FUNCIONAMIENTO:
  - RIGHT JOIN: Todas las filas de tabla derecha + coincidencias izquierda
  - FULL OUTER: Todas las filas de ambas tablas (coincidencias + no coincidencias)

✓ INTEGRACIÓN DE DATOS:
  - RIGHT JOIN detecta registros huérfanos en tabla derecha
  - FULL OUTER detecta inconsistencias en ambas direcciones
  - Útiles para auditoría y conciliación de datos

✓ DIFERENCIAS CLAVE vs SESIÓN 2:
  - LEFT JOIN: Todos izquierda, opcional derecha
  - RIGHT JOIN: Todos derecha, opcional izquierda
  - FULL OUTER: Todos ambas tablas, opcionalmente coincidentes

✓ LIMITACIONES TÉCNICAS:
  - PostgreSQL, SQL Server, Oracle: Soportan FULL OUTER nativamente
  - MySQL, SQLite: Emular con UNION de LEFT + RIGHT

✓ MANEJO DE NULLs:
  - RIGHT JOIN: NULLs en tabla izquierda = registros huérfanos
  - FULL OUTER: NULLs en ambas columnas = falta de coincidencia
  - Usar COALESCE para mejorar legibilidad de reportes
"""

print(summary)
print("="*70)

## SLIDE 25: Preguntas de Cierre

In [ ]:
print("\n❓ PREGUNTAS DE CIERRE:\n")

preguntas = [
    {
        "q": "1. ¿Cuál es la diferencia entre LEFT JOIN y RIGHT JOIN?",
        "r": "LEFT preserva todos de tabla izquierda. RIGHT preserva todos de tabla derecha."
    },
    {
        "q": "2. ¿Cuándo conviene usar RIGHT JOIN sobre LEFT JOIN?",
        "r": "Cuando la tabla derecha es la principal (ej: auditar carga de evaluaciones)."
    },
    {
        "q": "3. ¿Qué desafíos presenta FULL OUTER JOIN en MySQL/SQLite?",
        "r": "No hay soporte nativo, hay que emular con UNION de LEFT + RIGHT."
    },
    {
        "q": "4. ¿Cómo detectas registros huérfanos con RIGHT JOIN?",
        "r": "Donde IS NULL en columnas de tabla izquierda después del RIGHT JOIN."
    },
    {
        "q": "5. ¿Por qué usar FULL OUTER para auditoría?",
        "r": "Detecta ambos: registros huérfanos Y registros sin coincidencia al mismo tiempo."
    }
]

for item in preguntas:
    print(item["q"])
    print(f"  💭 {item['r']}\n")

## Cierre

In [ ]:
conn.close()
print("✓ Conexión cerrada")
print("\n¡Fin de la Sesión 3!")
print("\n📚 En la siguiente sesión: Consultas Agrupadas con GROUP BY y funciones de agregación")